<a href="https://colab.research.google.com/github/FranciscoBPereira/AnaliseDados_MEI_2526/blob/main/AD2526_P5B_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup, Version check and Common imports

# Python ≥ 3.7 is required
import sys
assert sys.version_info >= (3, 7)


# TensorFlow ≥ 2.8 is required
import tensorflow as tf
from packaging import version

assert version.parse(tf.__version__) >= version.parse("2.8.0")

# Common imports
import numpy as np
import os
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

print('Python version: ', sys.version_info)
print('TF version: ', tf.__version__)
print('Keras version: ', keras.__version__)
print('GPU is', 'available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE')

**1. Download a Segmentation Dataset: The Oxford-IIIT Pet Dataset**

In [ ]:
# The Oxford-IIIT Pet Dataset
# https://www.robots.ox.ac.uk/~vgg/data/pets/

!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
!tar -xf images.tar.gz
!tar -xf annotations.tar.gz
!rm images.tar.gz
!rm annotations.tar.gz


In [ ]:
import pathlib

input_dir = pathlib.Path("images")
target_dir = pathlib.Path("annotations/trimaps")

input_img_paths = sorted(input_dir.glob("*.jpg"))
# Ignores some spurious files in the trimaps directory that start with
# a "."
target_paths = sorted(target_dir.glob("[!.]*.png"))

In [ ]:
from keras.utils import load_img, img_to_array, array_to_img

# The original labels are 1 (foreground), 2 (background), 3 (contour)
# Here they are converted to 0, 127, 254 to enhance the visualization of the mask

def display_target(target_array):
    normalized_array = (target_array.astype("uint8") - 1) * 127
    plt.axis("off")
    plt.imshow(normalized_array[:, :, 0])

# Select one image from dataset. Change the index to check other examples
index = 9

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.title("Input Image")
plt.axis("off")
plt.imshow(load_img(input_img_paths[index]))

plt.subplot(1,2,2)
plt.title("Target Mask")
plt.axis("off")
mask = img_to_array(load_img(target_paths[index], color_mode="grayscale"))
plt.imshow(mask[:,:,0])

plt.show()

**2. Create Datasets**

In [ ]:
# Datasets are stored in numpy arrays

import random

img_size = (200, 200)
num_imgs = len(input_img_paths)

random.Random(1337).shuffle(input_img_paths)
random.Random(1337).shuffle(target_paths)

def path_to_input_image(path):
    return img_to_array(load_img(path, target_size=img_size))

def path_to_target(path):
    img = img_to_array(
        load_img(path, target_size=img_size, color_mode="grayscale")
    )
    img = img.astype("uint8") - 1
    return img

input_imgs = np.zeros((num_imgs,) + img_size + (3,), dtype="float32")
targets = np.zeros((num_imgs,) + img_size + (1,), dtype="uint8")
for i in range(num_imgs):
    input_imgs[i] = path_to_input_image(input_img_paths[i])
    targets[i] = path_to_target(target_paths[i])

In [ ]:
num_val_samples = 1000
train_input_imgs = input_imgs[:-num_val_samples]
train_targets = targets[:-num_val_samples]
val_input_imgs = input_imgs[-num_val_samples:]
val_targets = targets[-num_val_samples:]

In [ ]:
# Size of datasets

print('Training dataset: ', train_input_imgs.shape , train_targets.shape)
print('Validation dataset: ', val_input_imgs.shape , val_targets.shape)


**3. Create a Neural Network for Image Segmentation**

In [ ]:

inputs = keras.Input(shape=(200,200,3))

x = layers.Rescaling(1.0 / 255)(inputs)

x = layers.Conv2D(64, 3, strides=2, activation="relu", padding="same")(x)
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.Conv2D(128, 3, strides=2, activation="relu", padding="same")(x)
x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
x = layers.Conv2D(256, 3, strides=2, padding="same", activation="relu")(x)
x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)

x = layers.Conv2DTranspose(256, 3, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(256, 3, strides=2, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(128, 3, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(128, 3, strides=2, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(64, 3, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(64, 3, strides=2, activation="relu", padding="same")(x)

outputs = layers.Conv2D(3, 1, activation="softmax", padding="same")(x)

model = keras.Model(inputs=inputs, outputs=outputs)

model.summary()

In [ ]:
# Define the loss function

foreground_iou = keras.metrics.IoU(
    num_classes=3,
    target_class_ids=(0,),
    name="foreground_iou",
    sparse_y_true=True,
    sparse_y_pred=False,
)

**4. Model Training**

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=[foreground_iou],
)

history = model.fit(
    train_input_imgs,
    train_targets,
    epochs=20,
    batch_size=32,
    validation_data=(val_input_imgs, val_targets),
)

In [ ]:
epochs = range(1, len(history.history["foreground_iou"]) + 1)

train_iou = history.history["foreground_iou"]
val_iou = history.history["val_foreground_iou"]

plt.figure(figsize=(8,5))
plt.plot(epochs, train_iou, label="Training foreground IoU")
plt.plot(epochs, val_iou, label="Validation foreground IoU")
plt.title("Foreground IoU during training")
plt.xlabel("Epochs")
plt.ylabel("IoU")
plt.legend()
plt.show()

**5. Model Testing**

In [ ]:
# Test the effectiveness of the model on several validation images

i = 12
test_image = val_input_imgs[i]

mask = model.predict(np.expand_dims(test_image, 0))[0]

mask = np.argmax(mask, axis=-1)
mask = mask * 127

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.title("Original")
plt.axis("off")
plt.imshow(array_to_img(test_image))

plt.subplot(1,2,2)
plt.title("Predicted Mask")
plt.axis("off")
plt.imshow(mask)

plt.imshow(mask, cmap="viridis")

In [ ]:
# Upload a new image to the working directory
# Check how the model behaves

Image_Name = 'A.jpeg' #@param {type:"string"}

img = keras.utils.load_img(Image_Name, target_size=img_size)

test_image = keras.utils.img_to_array(img)

mask = model.predict(np.expand_dims(test_image, 0))[0]

mask = np.argmax(mask, axis=-1)
mask = mask * 127

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.title("Original")
plt.axis("off")
plt.imshow(array_to_img(test_image))

plt.subplot(1,2,2)
plt.title("Predicted Mask")
plt.axis("off")
plt.imshow(mask)

plt.imshow(mask, cmap="viridis")